# Homework 6 — Extending the pipeline

About 60 minutes. The first code cell rebuilds the bronze/silver/gold lake from session 6
(Colab forgets files between sessions). Then you extend the pipeline.

In [ ]:
# Run me first: finds the course data, or downloads it when running in Google Colab.
import subprocess
from pathlib import Path

COURSE_REPO = "https://github.com/alinafazliu/data-engineering-modul-1.git"


def find_data():
    for folder in [Path.cwd(), *Path.cwd().parents, Path("/content/course")]:
        for candidate in [folder / "data", folder / "month_course" / "data"]:
            if (candidate / "reference" / "tracks.csv").exists():
                return candidate


DATA = find_data()
if DATA is None:
    subprocess.run(["git", "clone", "--depth", "1", "--quiet", COURSE_REPO, "/content/course"], check=True)
    DATA = find_data()
print("Course data:", DATA)

In [ ]:
# Start Spark. Colab already has PySpark; anywhere else we install it first.
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("pyspark") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "pyspark>=3.5,<3.6"], check=True)
os.environ["PYSPARK_PYTHON"] = sys.executable  # Spark's workers must use the same Python as the notebook

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("wavely")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.ansi.enabled", "false")  # same behaviour on Spark 3.5 and 4.x: bad casts give null
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready")

In [ ]:
# Rebuilds the session 6 lake (bronze -> silver -> gold daily_country). Just run it.
LAKE = DATA.parent / "lake"
RAW_SCHEMA = (
    "event_id STRING, event_ts STRING, user_id STRING, track_id STRING, ms_played STRING, "
    "device STRING, country STRING, skipped STRING, shuffle STRING"
)
(
    spark.read.csv(str(DATA / "raw" / "plays_*.csv"), header=True, schema=RAW_SCHEMA)
    .withColumn("_source_file", F.col("_metadata.file_name"))
    .withColumn("_ingested_at", F.current_timestamp())
    .write.mode("overwrite").parquet(str(LAKE / "bronze" / "plays"))
)
typed = (
    spark.read.parquet(str(LAKE / "bronze" / "plays"))
    .withColumn("event_ts", F.to_timestamp("event_ts", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("ms_played", F.col("ms_played").cast("int"))
    .withColumn("skipped", F.col("skipped").cast("boolean"))
    .withColumn("shuffle", F.col("shuffle").cast("boolean"))
    .dropDuplicates(["event_id"])
)
is_valid = F.col("event_ts").isNotNull() & F.col("user_id").isNotNull() & (F.col("ms_played") >= 0)
(
    typed.filter(is_valid)
    .withColumn("play_date", F.to_date("event_ts"))
    .withColumn("minutes_played", F.round(F.col("ms_played") / 60000, 2))
    .write.mode("overwrite").partitionBy("play_date").parquet(str(LAKE / "silver" / "plays"))
)
typed.filter(~is_valid).write.mode("overwrite").parquet(str(LAKE / "silver" / "plays_quarantine"))

silver = spark.read.parquet(str(LAKE / "silver" / "plays"))
quarantine = spark.read.parquet(str(LAKE / "silver" / "plays_quarantine"))


def read_reference(name):
    return spark.read.csv(str(DATA / "reference" / f"{name}.csv"), header=True, inferSchema=True)


def check(name, bad_rows):
    n_bad = bad_rows.count()
    print(f"{'PASS' if n_bad == 0 else 'FAIL'}  {name}  ({n_bad} bad rows)")
    return n_bad == 0


print(silver.count(), "silver rows,", quarantine.count(), "quarantined")

### Exercise 1 — A new gold table: hour × device
The product team wants to know **when** people use **which** device.
Build `gold_hour_device` with columns `hour` (0-23), `device`, `plays`, `hours` (2 decimals) and write it to `LAKE / "gold" / "hour_device"`.
Then show the top 5 rows by `plays`.

In [ ]:
# YOUR CODE HERE

### Exercise 2 — Do people listen longer at the weekend?
For each user and day, sum the hours listened. Then compute the **average hours per active user per day**,
separately for weekdays and weekends (`F.dayofweek` 1 = Sunday, 7 = Saturday). Round to 2 decimals.

In [ ]:
# YOUR CODE HERE

### Exercise 3 — More quality checks
Write two checks with `check(name, bad_rows)`:
1. `country` is one of `AT, SK, HU, CZ, DE, PL`
2. `device` is one of `mobile, desktop, web, smart_speaker`

Then write a third check that you **expect to fail**: "every user in silver has at least 2 active days". How many users fail it? Is that a data problem or a wrong rule?

In [ ]:
# YOUR CODE HERE

### Exercise 4 — Investigate the quarantine
Add a column `reason` to the quarantined rows: `"negative ms_played"`, `"missing timestamp"`, `"missing user"` or `"other"`.
Count rows per `reason` and per `_source_file`. Is one day worse than the others?

In [ ]:
# YOUR CODE HERE

### Exercise 5 (Stretch) — Artist payouts
Wavely pays **€0.003 per stream** to the artist. A stream is a play that was **not skipped**.
1. Build `gold_artist_payouts`: `artist_name`, `streams`, `payout_eur` (2 decimals), sorted by payout.
2. Add a reconciliation check: the sum of `streams` in your table equals the number of non-skipped plays in silver.

In [ ]:
# YOUR CODE HERE